# SQL 9: deduplication

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-09-deduplication.ipynb)

Duplicates arrive from retries, replayed messages, late updates and dirty source systems. Deduplication is not one
operation. First decide what counts as **the same record** (the key). Then decide **which copy wins** (the rule).
Without a deterministic rule, the result can change between runs.

This chapter shows how to find duplicates, keep one row per key, and apply updates. It ends with practice questions
and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'ava@x.com'), (2, 'ben@x.com'), (3, 'ava@x.com'), (4, ' Ava@X.com ')
AS contacts(id, email)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW profile_updates AS
SELECT * FROM VALUES
  (1, 1, 'old@x.com', DATE '2023-01-01'),
  (2, 1, 'new@x.com', DATE '2023-06-01'),
  (3, 1, 'mid@x.com', DATE '2023-03-01'),
  (4, 2, 'b@x.com',   DATE '2023-02-01'),
  (5, 2, 'b2@x.com',  DATE '2023-02-01')
AS profile_updates(update_id, customer_id, email, updated_at)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW limits AS
SELECT * FROM VALUES (1, 1000), (2, 2000), (3, 1500), (4, 500), (5, 800), (6, 900), (7, 300), (8, 400)
AS limits(customer_id, credit_limit)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW limit_updates AS
SELECT * FROM VALUES
  (1, 1200, DATE '2023-05-01'),
  (1, 1500, DATE '2023-08-01'),
  (4, 700,  DATE '2023-04-01'),
  (7, 350,  DATE '2023-02-01')
AS limit_updates(customer_id, credit_limit, updated_at)
""")

### The data

- `contacts` (4 rows): email addresses typed by hand. Rows 1 and 3 hold the identical string `'ava@x.com'`; row 4
  holds `' Ava@X.com '`, the same mailbox with spaces and capitals.
- `profile_updates` (5 rows): an append-only feed of email changes. Customer 1 has three versions on three dates.
  Customer 2 has two versions on the **same** date, which is the case that breaks careless deduplication.
- `limits` (8 rows): the current credit limit of customers 1 to 8, one row each.
- `limit_updates` (4 rows): changes to apply to `limits`. Customer 1 has two changes, customers 4 and 7 have one,
  and the other five customers have none.

## 1. Find duplicates

Two terms first:

- The **key** is the column, or set of columns, that should identify one real-world thing. For a contact list the
  key is the email address.
- The **grain** of a table is what one row stands for, for example "one row per email address". A duplicate is a
  row that breaks the grain: a second row for a key that already has one.

`GROUP BY key HAVING COUNT(*) > 1` lists the keys that appear more than once. `GROUP BY` makes one group per key
value, `COUNT(*)` counts the rows in each group, and `HAVING` keeps only the groups with more than one row.

### Where you'll use it

- **Data-quality checks:** a test that fails when a key that should be unique is not.
- **Investigating a row-count jump:** before fixing anything, find which keys gained extra rows.

### Normalise the key first

The comparison is exact, character by character. `'ava@x.com'` and `' Ava@X.com '` differ in spaces and case, so a
plain `GROUP BY email` puts them in different groups and finds only 2 copies. Email addresses are not case
sensitive in practice and the spaces are typing errors, so the key should be **normalised** before grouping:
`trim` removes leading and trailing spaces, and `lower` converts to lower case. All three Ava rows then share the key
`'ava@x.com'`.

| id | email           | lower(trim(email)) |
|----|-----------------|--------------------|
| 1  | `ava@x.com`     | `ava@x.com`        |
| 2  | `ben@x.com`     | `ben@x.com`        |
| 3  | `ava@x.com`     | `ava@x.com`        |
| 4  | `' Ava@X.com '` | `ava@x.com`        |

In [ ]:
raw_q = """
SELECT email, COUNT(*) AS copies
FROM contacts
GROUP BY email
HAVING COUNT(*) > 1
"""
spark.sql(raw_q).show()
assert spark.sql(raw_q).collect() == [("ava@x.com", 2)]

normalised_q = """
SELECT lower(trim(email)) AS normalised_email, COUNT(*) AS copies
FROM contacts
GROUP BY lower(trim(email))
HAVING COUNT(*) > 1
"""
spark.sql(normalised_q).show()
assert spark.sql(normalised_q).collect() == [("ava@x.com", 3)]

### Seeing every copy

`GROUP BY` collapses each key to one row, so it hides which rows are the copies. A window function keeps every row
and adds the group's count next to it. `COUNT(*) OVER (PARTITION BY ...)` counts the rows in the row's
**partition**, the set of rows that share the same value of the `PARTITION BY` expression. Filtering on that count
lists the duplicate rows themselves, with their ids.

In [ ]:
q = """
SELECT id, email
FROM (
  SELECT id, email, COUNT(*) OVER (PARTITION BY lower(trim(email))) AS copies
  FROM contacts
)
WHERE copies > 1
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "ava@x.com"), (3, "ava@x.com"), (4, " Ava@X.com ")]

## 2. Keep one row per key

`profile_updates` has several versions per customer. To keep the newest one, number the rows inside each customer
and keep row 1:

1. `PARTITION BY customer_id` splits the rows into one partition per customer.
2. `ORDER BY updated_at DESC, update_id DESC` sorts each partition, newest first.
3. `ROW_NUMBER()` gives the rows 1, 2, 3, ... in that order, restarting at 1 in each partition.
4. The outer `WHERE rn = 1` keeps the first row of each partition.

Worked by hand:

| customer_id | update_id | updated_at | rn |
|-------------|-----------|------------|----|
| 1           | 2         | 2023-06-01 | 1  |
| 1           | 3         | 2023-03-01 | 2  |
| 1           | 1         | 2023-01-01 | 3  |
| 2           | 5         | 2023-02-01 | 1  |
| 2           | 4         | 2023-02-01 | 2  |

Customer 2's two rows have the same date, so the date alone cannot order them. `update_id DESC` is the
**tie-breaker**: a second sort column that decides between rows the first column ranks equally. Here the higher id
wins. Because `update_id` is unique, the order is fully decided and the same row wins on every run. A query with
that property is called **deterministic**.

### Where you'll use it

- **Replayed messages:** a Kafka consumer or CDC feed delivers the same change twice; a window function keeps one version.
- **Latest state from a history table:** the current address of each customer from an address-change log.

In [ ]:
latest_q = """
SELECT customer_id, update_id, email
FROM (
  SELECT customer_id, update_id, email,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, update_id DESC) AS rn
  FROM profile_updates
)
WHERE rn = 1
ORDER BY customer_id
"""
spark.sql(latest_q).show()
assert spark.sql(latest_q).collect() == [(1, 2, "new@x.com"), (2, 5, "b2@x.com")]

### Why the tie-breaker is needed

Without the tie-breaker, the sort key is `updated_at` alone, and customer 2 has two rows with the same value.
`ROW_NUMBER` must still number them 1 and 2, but SQL does not say which row gets 1. Spark may pick either, and the
choice can change when the data is split across workers differently. The pipeline would then flip between
`b@x.com` and `b2@x.com` from one run to the next.

`RANK()` makes the tie visible: it gives rows with equal sort values the same rank. Two rows of customer 2 share
rank 1, which proves that `updated_at` does not decide a single winner.

In [ ]:
q = """
SELECT customer_id, update_id,
       RANK() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rnk
FROM profile_updates
ORDER BY customer_id, update_id
"""
spark.sql(q).show()
tied_for_first = [(r.customer_id, r.update_id) for r in spark.sql(q).collect() if r.rnk == 1]
assert tied_for_first == [(1, 2), (2, 4), (2, 5)]

### When DISTINCT is enough

`DISTINCT` removes rows that are equal in **every** selected column. It is the right tool when the copies are
exact, for example a message delivered twice with identical content. It cannot pick between versions that differ
in some column: customer 1's three versions have different emails, so `DISTINCT` keeps all three.

In [ ]:
exact_q = """
SELECT DISTINCT customer_id, email
FROM VALUES (1, 'a@x.com'), (1, 'a@x.com'), (2, 'b@x.com') AS replayed(customer_id, email)
ORDER BY customer_id
"""
assert spark.sql(exact_q).collect() == [(1, "a@x.com"), (2, "b@x.com")]

versions_q = "SELECT DISTINCT customer_id, email FROM profile_updates"
assert spark.sql(versions_q).count() == 5

### The GROUP BY MAX pitfall

A common alternative finds each customer's maximum `updated_at` and joins back to the rows with that timestamp. It
is correct only when the maximum is unique. Customer 2 has two rows at the maximum, so the join returns both, and
the result has two rows for one key: the duplicates are still there.

In [ ]:
q = """
SELECT p.customer_id, p.email
FROM profile_updates p
JOIN (
  SELECT customer_id, MAX(updated_at) AS latest
  FROM profile_updates
  GROUP BY customer_id
) m ON m.customer_id = p.customer_id AND m.latest = p.updated_at
ORDER BY p.customer_id, p.email
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "new@x.com"), (2, "b2@x.com"), (2, "b@x.com")]

## 3. Delete duplicates by keeping the survivors

Temporary views cannot be modified, so you cannot `DELETE` duplicates from one. The equivalent is to select the
rows to keep, the **survivors**.

Say that row `k` **beats** row `p` when both belong to the same customer and `k` is newer: a later `updated_at`, or
the same `updated_at` and a higher `update_id`. This is the same order the `ROW_NUMBER` query used. A row survives
when no row beats it, which `NOT EXISTS` expresses directly: keep `p` when the subquery finds no `k` that beats it.

Worked by hand: update 1 is beaten by updates 2 and 3, update 3 by update 2, and update 4 by update 5 (same date,
higher id). Updates 2 and 5 are never beaten, so they survive.

### Where you'll use it

- **Repairing a table:** rebuilding a table with only the survivors, instead of deleting rows one by one.
- **Reviewing a delete before running it:** the "beaten" predicate can be run as a `SELECT` first to list exactly
  which rows a `DELETE` would remove.

In [ ]:
latest_q = """
SELECT customer_id, update_id, email
FROM (
  SELECT customer_id, update_id, email,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, update_id DESC) AS rn
  FROM profile_updates
)
WHERE rn = 1
ORDER BY customer_id
"""

survivors_q = """
SELECT p.update_id, p.customer_id, p.email
FROM profile_updates p
WHERE NOT EXISTS (
  SELECT 1 FROM profile_updates k
  WHERE k.customer_id = p.customer_id
    AND (k.updated_at > p.updated_at
         OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
)
ORDER BY p.update_id
"""
spark.sql(survivors_q).show()
assert spark.sql(survivors_q).collect() == [(2, 1, "new@x.com"), (5, 2, "b2@x.com")]

survivor_ids = {r.update_id for r in spark.sql(survivors_q).collect()}
row_number_ids = {r.update_id for r in spark.sql(latest_q).collect()}
assert survivor_ids == row_number_ids

On a database table that allows deletes, such as PostgreSQL, the same predicate with `EXISTS` in place of
`NOT EXISTS` removes the beaten rows. This is reference SQL; it cannot run on a temporary view:

```sql
DELETE FROM profile_updates p
WHERE EXISTS (
  SELECT 1 FROM profile_updates k
  WHERE k.customer_id = p.customer_id
    AND (k.updated_at > p.updated_at
         OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
);
```

## 4. Apply updates to a base table

To update a base table from a feed, two steps are needed:

1. Reduce the feed to **one row per key**: the latest update for each customer. Customer 1 has two updates; only
   the August one (1500) should apply.
2. `LEFT JOIN` the base table to that reduced feed, and use `COALESCE(new, old)`. `COALESCE` returns its first
   argument that is not NULL. A customer with an update gets the new value; a customer without one has a NULL from
   the left join, so `COALESCE` falls back to the current value.

Expected result, by hand: customer 1 becomes 1500, customer 4 becomes 700, customer 7 becomes 350, and the other
five keep their limits.

The `ROW_NUMBER` below orders by `updated_at` only. That is deterministic here because no customer has two updates
on the same date, and the first assert checks that before relying on it.

### Where you'll use it

- **Applying a CDC batch:** merging a day of changes into a dimension table. This is the core of most incremental warehouse loads.
- **Snapshot refresh:** rebuilding the current-state table from yesterday's snapshot plus today's changes.

In [ ]:
same_day_updates = spark.sql("""
SELECT customer_id, updated_at FROM limit_updates
GROUP BY customer_id, updated_at
HAVING COUNT(*) > 1
""")
assert same_day_updates.count() == 0

spark.sql("""
CREATE OR REPLACE TEMP VIEW latest_limit_updates AS
SELECT customer_id, credit_limit
FROM (
  SELECT customer_id, credit_limit,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
  FROM limit_updates
)
WHERE rn = 1
""")

applied_q = """
SELECT b.customer_id, COALESCE(l.credit_limit, b.credit_limit) AS credit_limit
FROM limits b
LEFT JOIN latest_limit_updates l ON l.customer_id = b.customer_id
ORDER BY b.customer_id
"""
spark.sql(applied_q).show()
assert spark.sql(applied_q).collect() == [
    (1, 1500), (2, 2000), (3, 1500), (4, 700), (5, 800), (6, 900), (7, 350), (8, 400)
]

### Three ways to get it wrong

Each mistake below changes the result in a way that is easy to miss:

- **No `COALESCE`.** The five customers without an update get NULL. A whole column is silently wiped for every key
  the batch did not touch.
- **Inner join instead of left join.** Customers without an update disappear: the table shrinks from 8 rows to 3.
- **Joining the raw feed.** Without step 1, customer 1 matches two update rows, so the join returns two rows for
  customer 1. The join multiplies rows and creates the duplicates this chapter is about.

In [ ]:
no_coalesce = spark.sql("""
SELECT b.customer_id, l.credit_limit
FROM limits b
LEFT JOIN latest_limit_updates l ON l.customer_id = b.customer_id
""")
assert no_coalesce.where("credit_limit IS NULL").count() == 5

inner = spark.sql("""
SELECT b.customer_id
FROM limits b
JOIN latest_limit_updates l ON l.customer_id = b.customer_id
""")
assert inner.count() == 3

raw_feed = spark.sql("""
SELECT b.customer_id
FROM limits b
LEFT JOIN limit_updates u ON u.customer_id = b.customer_id
""")
assert raw_feed.count() == 9
assert raw_feed.where("customer_id = 1").count() == 2

## 5. On a table format that supports it

Delta Lake and Apache Iceberg tables can be changed in place. The update above becomes one `MERGE` statement, an
**upsert**: update the rows whose key already exists, insert the rows whose key is new.

- `ON` names the key that links a source row to a target row.
- `WHEN MATCHED` runs for target rows that have a matching source row.
- `WHEN NOT MATCHED` runs for source rows with no target row.

The source must hold at most one row per key, so it is the reduced `latest_limit_updates`, not the raw feed. If two
source rows match one target row, the engine cannot tell which update to apply, and Delta Lake stops the `MERGE`
with an error. Keys that are not in the source are untouched, which is the job `COALESCE` did in section 4.

This cell is a reference only, because it needs such a table:

```sql
MERGE INTO limits AS target
USING latest_limit_updates AS source
  ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET target.credit_limit = source.credit_limit
WHEN NOT MATCHED THEN INSERT (customer_id, credit_limit) VALUES (source.customer_id, source.credit_limit)
```

### Where you'll use it

- **Lakehouses:** Delta Lake, Iceberg and Hudi provide `MERGE` for exactly this upsert on data lake files.

## Interview notes

- **DISTINCT vs GROUP BY vs ROW_NUMBER.** `DISTINCT` removes exact duplicate rows across all selected columns.
  `GROUP BY` removes duplicates on a key and aggregates the rest. `ROW_NUMBER` keeps one chosen row per key, which is
  what most pipelines need.
- **ROW_NUMBER vs RANK.** `ROW_NUMBER` always gives one row number 1 per partition. `RANK` gives tied rows the same
  rank, so `rank = 1` can keep two rows. Use `ROW_NUMBER` with a unique tie-breaker for deduplication.
- **Be idempotent.** An idempotent job gives the same result whether it runs once or twice. A deterministic
  tie-breaker and an overwrite or merge write make that true; an append does not.
- **Reduce the source before you join or merge.** One row per key in the update feed prevents a join from
  multiplying rows and a `MERGE` from failing.
- **Duplicates are a symptom.** Ask where they come from: retries, a join that multiplies rows, or an ingestion
  that is not keyed. Fix the source when you can.

## Summary

- Define the key, then define a deterministic winner. Add a unique tie-breaker.
- Normalise keys before comparing them.
- `DISTINCT` handles only exact copies; `ROW_NUMBER` picks one version per key.
- "Delete duplicates" means "keep the survivors". Express it as a predicate you can check before you run it.
- When applying updates, reduce the feed to one row per key, and guard with a left join and `COALESCE` so untouched
  keys keep their values.

## Practice questions

1. Which email addresses appear more than once after normalising with `lower(trim(...))`? Show the normalised email and the count.
2. Keep the **earliest** version per customer, breaking ties on the lower `update_id`. Show `customer_id`, `update_id` and `email`, sorted by `customer_id`.
3. How many rows of `profile_updates` are not survivors, that is, are beaten by another row of the same customer
   (a later `updated_at`, or the same `updated_at` and a higher `update_id`)?
4. After applying the latest update per customer from `limit_updates`, which customers' credit limit changed? Show
   `customer_id`, the old limit and the new limit, sorted by `customer_id`.

## Answers

### Question 1

Group by the normalised email and keep the groups with more than one row.

In [ ]:
q = """
SELECT lower(trim(email)) AS normalised_email, COUNT(*) AS copies
FROM contacts
GROUP BY lower(trim(email))
HAVING COUNT(*) > 1
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("ava@x.com", 3)]

### Question 2

Reverse both sort directions so the oldest row, and on a tie the lower id, gets row number 1. Customer 2's tie on
2023-02-01 now goes to update 4.

In [ ]:
q = """
SELECT customer_id, update_id, email
FROM (
  SELECT customer_id, update_id, email,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at ASC, update_id ASC) AS rn
  FROM profile_updates
)
WHERE rn = 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1, "old@x.com"), (2, 4, "b@x.com")]

### Question 3

Count the rows for which a beating row exists: `EXISTS` instead of `NOT EXISTS`. Updates 1, 3 and 4 are beaten.
Survivors plus non-survivors must equal the 5 rows of the table.

In [ ]:
q = """
SELECT COUNT(*) AS not_survivors
FROM profile_updates p
WHERE EXISTS (
  SELECT 1 FROM profile_updates k
  WHERE k.customer_id = p.customer_id
    AND (k.updated_at > p.updated_at
         OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3,)]
assert spark.sql(q).collect()[0][0] + spark.sql(survivors_q).count() == spark.table("profile_updates").count()

### Question 4

Join `limits` to the one-row-per-key `latest_limit_updates` with an inner join, because only customers with an
update can change, and keep the rows where the value differs.

In [ ]:
q = """
SELECT b.customer_id, b.credit_limit AS old_limit, l.credit_limit AS new_limit
FROM limits b
JOIN latest_limit_updates l ON l.customer_id = b.customer_id
WHERE l.credit_limit <> b.credit_limit
ORDER BY b.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1000, 1500), (4, 500, 700), (7, 300, 350)]